# B2 국산화개발품목 · A7 국외조달 조달계획 정제 → `clean_` 적재

작성 2026-09-17 (검토 보고서 2026-09-16 §6 작업 2). 대상은 판단이 거의 필요 없는 두 테이블이다.

| 원본 파일(데이터셋 키) | 정제 결과(clean_) | 기대 행 수 | 규칙 근거 |
|---|---|---|---|
| `raw_dapa_localized_item` 33,965 | `clean_dapa_localized_item` — 사업명×부품관리번호 고유, `dup_count` 보존 | 25,025 | `docs/idea-review.md` §5 B2 적용 규칙 ③, `docs/db/schema-design.md` §3-5 |
| `raw_dapa_overseas_plan` 3,029 | `clean_dapa_overseas_plan` — 판단번호 단위, 전자 관련 후보 플래그(검수 전) | 3,024 이하 | `docs/report/feedback/professor-feedback-2026-09-15.md` §2, `idea-review.md` §3-15 |

원칙
- 원본은 `data/raw/` 파일을 `read_raw()`로 읽는다(RDS `raw_` 표는 2026-09-22 삭제 — `raw_row_id` = 파서 순번, 파일명 정렬 × 행 순). 정제 결과는 `clean_`에만 쓰고, 대상 테이블이 **비어 있을 때만** 적재한다(재적재는 `db/reset_data.sql` 방식).
- 판단 속성은 기본값으로 둔다: `electronics_review_status`는 `미검수`. HS6 품목군 대응(`category`·`category_link_status`)은 **2026-09-21 카테고리 맵 폐기로 열 자체를 없앴다**(B2는 FSC 축에서만).
- 전자 관련 후보는 `대표품명` 키워드 일치일 뿐이다. 검수 전 값은 화면·문서에서 "잠정"으로만 쓴다.
- 단계별 건수는 `meta_load_log`에 기록한다(`원본 전체 → 중복 처리 후 → 관련 후보`).
- 접속 정보는 `.env`(`scripts/load_db.py`의 `connect()` 재사용). 비밀번호는 출력하지 않는다.

실행: 저장소 루트에서 `python -m jupyter nbconvert --to notebook --execute --inplace notebooks/clean_b2_a7.ipynb`


In [1]:
import sys, re
from pathlib import Path
from datetime import datetime
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from load_db import connect, log_stage, current_user, read_raw, log_raw_stage   # .env 읽기 · meta_load_log 기록 · 원본 파일 읽기
RAW = {}   # 원본 파일 DataFrame 캐시 {데이터셋 키: read_raw 결과}. RDS raw_ 표는 2026-09-22 삭제 — row_id = 파서 순번 = clean.raw_row_id
def raw(table, cols=None):
    if table not in RAW:
        RAW[table] = read_raw(table)
    df = RAW[table]
    return (df[["row_id"] + [c for c in cols if c != "row_id"]] if cols else df).copy()

NB_TAG = "notebooks/clean_b2_a7.ipynb"
conn = connect()
cur = conn.cursor()
USER = current_user(cur)
def q(sql):
    cur.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
print("connected as", USER, "| pandas", pd.__version__)

connected as etl_rw@% | pandas 3.0.5


## 1. B2 국산화개발품목 → `clean_dapa_localized_item`

규칙
- 키 = `project_name` × `part_mgmt_no`. 같은 키의 원본 행 수를 `dup_count`에 남기고 대표 행은 `row_id`가 가장 작은 행(`first_raw_row_id`).
- 같은 키인데 내용(NSN·FSC·품명·계약업체·최종수정일)이 다른 행이 있으면 **충돌**로 세어 출력한다. 대표 행 값을 쓰되 건수를 보고한다.
- `fsc4` = 4자리 숫자일 때만, 아니면 NULL(`0`·공란). `fsc2` = 앞 2자리. `is_electronic_group` = `fsc2 IN (58, 59)`.
- HS6 품목군 대응은 하지 않는다(2026-09-21 카테고리 맵 폐기 — `ref_category_map`·`category`·`category_link_status` 삭제). B2는 FSC 축(화면 ②)에서만 본다.
- `contractor_name_norm`: 공백 정리 + 법인격 표기(`(주)`·`㈜`·`주식회사`·`(유)`·`유한회사`) 제거만. 업체 연결은 이 노트북 범위 밖.
- `last_modified_date`는 스냅샷 날짜이며 연도 축에 쓰지 않는다(§3-14).


In [2]:
raw_b2 = raw("raw_dapa_localized_item", ["project_name", "part_mgmt_no", "nsn", "fsc", "item_name", "contractor_name", "last_modified_date"])
print("원본 전체:", len(raw_b2))

key = ["project_name", "part_mgmt_no"]
content_cols = ["nsn", "fsc", "item_name", "contractor_name", "last_modified_date"]
grp = raw_b2.groupby(key, sort=False)
b2 = grp.first().reset_index()                     # 대표 행 = row_id 최소(정렬 유지)
b2["dup_count"] = grp.size().values
b2["first_raw_row_id"] = grp["row_id"].min().values
n_variants = grp[content_cols].nunique(dropna=False).max(axis=1)   # 같은 키 안에서 내용 조합이 2가지 이상이면 충돌
b2["has_conflict"] = (n_variants.values > 1)
print("중복 처리 후(사업×부품 고유):", len(b2), "| 완전 중복으로 줄어든 행:", len(raw_b2) - len(b2))
print("dup_count 분포:", b2["dup_count"].value_counts().sort_index().to_dict())
print("같은 키 내용 충돌 키 수:", int(b2["has_conflict"].sum()))
print("고유 부품(part_mgmt_no):", b2["part_mgmt_no"].nunique(), "| 사업 수:", b2["project_name"].nunique())

원본 전체: 33965
중복 처리 후(사업×부품 고유): 25025 | 완전 중복으로 줄어든 행: 8940
dup_count 분포: {1: 20703, 2: 2997, 3: 498, 4: 396, 5: 144, 6: 51, 7: 47, 8: 32, 9: 23, 10: 20, 11: 14, 12: 17, 13: 14, 14: 9, 15: 9, 16: 4, 17: 7, 18: 5, 19: 1, 20: 3, 21: 4, 22: 2, 23: 1, 25: 2, 26: 2, 27: 3, 28: 2, 31: 1, 33: 1, 35: 3, 36: 2, 37: 1, 43: 1, 47: 1, 48: 1, 49: 1, 52: 1, 55: 1, 57: 1}
같은 키 내용 충돌 키 수: 0
고유 부품(part_mgmt_no): 12788 | 사업 수: 28


In [3]:
b2["fsc4"] = b2["fsc"].where(b2["fsc"].astype(str).str.fullmatch(r"[0-9]{4}"), None)
b2["fsc2"] = b2["fsc4"].str[:2]
b2["is_electronic_group"] = b2["fsc2"].isin(["58", "59", "60"]).astype(int)
print("fsc4 NULL(4자리 아님):", int(b2["fsc4"].isna().sum()), "| 58·59·60 군급 행(60 해당 0):", int(b2["is_electronic_group"].sum()))

# (HS6 품목군 대응 category·category_link_status 산출은 2026-09-21 카테고리 맵 폐기로 삭제)

_legal = re.compile(r"\(주\)|㈜|주식회사|\(유\)|유한회사|\(재\)|재단법인|\(사\)|사단법인")
def norm_name(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    t = _legal.sub(" ", str(s))
    t = re.sub(r"\s+", " ", t).strip()
    return t or None
b2["contractor_name_norm"] = b2["contractor_name"].map(norm_name)
lmd = pd.to_datetime(b2["last_modified_date"], errors="coerce")
b2["last_modified_date"] = [None if pd.isna(x) else x.date() for x in lmd]
print("최종수정일 NULL:", sum(x is None for x in b2["last_modified_date"]))

fsc4 NULL(4자리 아님): 16 | 58·59·60 군급 행(60 해당 0): 2717


최종수정일 NULL: 18160


## 2. A7 국외조달 조달계획 → `clean_dapa_overseas_plan`

규칙
- 키 = `decision_no`(판단번호, DDL의 PK). 같은 판단번호의 원본 행 수를 `dup_count`, 내용이 다르면 `has_conflict=1`. 대표 행은 `row_id` 최소.
  - 원본에서 같은 판단번호 5쌍은 완전 중복이 아니라 **집행예정월·진행상태·(일부는 품명)이 다른 별개 계획 행**이다. PK 때문에 한 행만 남기므로 **연도별 예산 합은 원본 3,029행 집계와 5행만큼 다르다** — 배경 ⓪ 화면에서 원본 전체 합(18.26조 원)을 쓸 때는 `raw_` 직접 집계(`table-guide.md` §5 SQL 4)를 쓰고, 이 표는 판단번호 단위 값임을 라벨에 적는다.
- `plan_year`·`plan_month`는 `집행예정월`(`YYYY-MM-DD`)에서, `budget_krw`는 정수. 판단번호·집행유형·예산·집행예정월이 없는 행은 NOT NULL을 만족하지 못하므로 제외하고 건수·사유를 기록한다.
- `is_contracted` = `progress_status IN ('계약','부분계약')`. 그 외는 공고 진행 중이므로 "집행 예산"이 아니다(§3-15).
- `is_electronics_candidate` = `대표품명` 키워드 일치(아래 목록, 검수 전 **잠정**). `electronics_review_status`는 전부 `미검수`. `system_family_hint`는 키워드 계열이며 품목군(HS6) 대응이 아니다.
- 키워드는 `professor-feedback-2026-09-15.md` §2 목록(전자·통신·레이더·반도체·회로·센서·증폭·모듈·안테나·항법·광학·음탐)에 소나·송수신·데이터링크·전파·GPS·IFF를 더했다. `EO`처럼 짧은 영문 약어는 오탐(`EOD 슈트`)이 확인돼 넣지 않는다.


In [4]:
raw_a7 = raw("raw_dapa_overseas_plan", ["plan_month", "decision_no", "rep_item_name", "exec_type", "contract_method", "exec_agency", "budget_amount", "progress_status"])
print("원본 전체:", len(raw_a7), "| 원본 예산 합(억원):", round(pd.to_numeric(raw_a7["budget_amount"], errors="coerce").sum() / 1e8))
bad = raw_a7[raw_a7[["decision_no", "plan_month", "exec_type", "budget_amount"]].isna().any(axis=1)]
print("필수 값 결측으로 제외:", len(bad))
display(bad[["row_id", "plan_month", "decision_no", "rep_item_name", "exec_type", "budget_amount", "progress_status"]])
a7 = raw_a7.drop(bad.index).copy()

content_cols = ["plan_month", "rep_item_name", "exec_type", "contract_method", "exec_agency", "budget_amount", "progress_status"]
g = a7.groupby("decision_no", sort=False)
p = g.first().reset_index()
p["dup_count"] = g.size().values
p["first_raw_row_id"] = g["row_id"].min().values
p["has_conflict"] = (g[content_cols].nunique(dropna=False).max(axis=1).values > 1).astype(int)
dups = a7[a7["decision_no"].isin(p.loc[p["dup_count"] > 1, "decision_no"])].sort_values(["decision_no", "row_id"])
dropped_budget = pd.to_numeric(dups["budget_amount"]).sum() - pd.to_numeric(p.loc[p["dup_count"] > 1, "budget_amount"]).sum()
print("판단번호 단위:", len(p), "| 중복 판단번호:", int((p["dup_count"] > 1).sum()), "| 그중 내용 충돌:", int(p["has_conflict"].sum()),
      "| PK 축약으로 빠지는 예산(억원):", round(dropped_budget / 1e8, 1))
display(dups[["decision_no", "row_id", "plan_month", "rep_item_name", "exec_type", "budget_amount", "progress_status"]])

원본 전체: 3029 | 원본 예산 합(억원): 182607
필수 값 결측으로 제외: 1


,row_id,plan_month,decision_no,rep_item_name,exec_type,budget_amount,progress_status
767,768,2018-11-30,TST00001001,라지트부속,None,252507879,2차공고의뢰중


판단번호 단위: 3023 | 중복 판단번호: 5 | 그중 내용 충돌: 5 | PK 축약으로 빠지는 예산(억원): 23.6


,decision_no,row_id,plan_month,rep_item_name,exec_type,budget_amount,progress_status
1383,BBBB0001001,1384,2020-06-30,쌍안경(VECTOR 21) 부품정비,장비정비,358228466,계약
1697,BBBB0001001,1698,2021-05-31,EOD 로보트,장비정비,358228466,판단완료
1482,DDEB0159001,1483,2020-10-03,정밀접근레이더(RP-5GI/M) 기술지원,기술용역,372412880,계약
2520,DDEB0159001,2521,2024-05-31,정밀접근레이더(RP-5GI/M) 기술지원,기술용역,372412880,계약
2573,PABB4006001,2574,2024-08-30,잠수함 레이더 부품정비,장비정비,185453151,판단완료
2681,PABB4006001,2682,2024-12-20,잠수함 레이더 부품정비,장비정비,205453261,계약
1710,PAEB1006001,1711,2021-06-30,수중생환 훈련장비 기술용역,기술용역,89837660,판단중
2405,PAEB1006001,2406,2023-12-10,수중생환 훈련장비 기술용역,기술용역,89837660,판단중
2643,PAMH4008001,2644,2024-10-31,해상초계기 부품구매 한도액계약,한도액부품,1332951373,판단완료
2658,PAMH4008001,2659,2024-12-10,해상초계기 부품구매 한도액계약,한도액부품,1332951373,계약


In [5]:
p["plan_year"] = p["plan_month"].str[:4]
p["plan_month"] = p["plan_month"].str[:7]
p["budget_krw"] = pd.to_numeric(p["budget_amount"], errors="coerce").astype("int64")
p["is_contracted"] = p["progress_status"].isin(["계약", "부분계약"]).astype(int)

KW = {
    "레이더": r"레이더|RADAR",
    "통신": r"통신|송수신|데이터링크|안테나|전파|IFF|무전기",
    "항법": r"항법|GPS|INS",
    "전자광학": r"광학|열상|열영상|야시|적외선|EO/IR|EOTS",
    "음탐": r"음탐|소나|SONAR",
    "전자부품": r"반도체|회로|증폭|모듈|센서",
    "전자(기타)": r"전자|전기|디스플레이|컴퓨터|프로세서|신호처리",
}
def family(name):
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return None
    for fam, pat in KW.items():
        if re.search(pat, str(name), flags=re.IGNORECASE):
            return fam
    return None
p["system_family_hint"] = p["rep_item_name"].map(family)
p["is_electronics_candidate"] = p["system_family_hint"].notna().astype(int)
p["electronics_review_status"] = "미검수"
n_cand = int(p["is_electronics_candidate"].sum())
print("전자 관련 후보(잠정, 검수 전):", n_cand, "/", len(p), f"= {n_cand/len(p)*100:.1f}% (건수 기준)")
print("계열별:", p["system_family_hint"].value_counts().to_dict())
print("예산 비중(잠정):", f"{p.loc[p.is_electronics_candidate==1,'budget_krw'].sum()/p['budget_krw'].sum()*100:.1f}%")
display(p.loc[p.is_electronics_candidate == 1, ["plan_year", "rep_item_name", "exec_type", "system_family_hint"]].sample(15, random_state=1))

전자 관련 후보(잠정, 검수 전): 392 / 3023 = 13.0% (건수 기준)
계열별: {'통신': 127, '레이더': 122, '전자(기타)': 63, '음탐': 38, '항법': 21, '전자광학': 17, '전자부품': 4}
예산 비중(잠정): 21.3%


,plan_year,rep_item_name,exec_type,system_family_hint
671,2018,P-3 RADAR 부품정비,장비정비,레이더
1606,2020,대포병탐지레이더 기술용역,기술용역,레이더
2763,2025,통신전자장비 부품,(확정)부품,통신
1218,2019,해군통신부품,(확정)부품,통신
2949,2025,장보고-Ⅲ BATCH-Ⅰ정비지원장비(컴퓨터시스템),장비,전자(기타)
2042,2022,함정용 대공레이더(SMART-L) 부품정비,장비정비,레이더
636,2018,관성항법장비 부품정비,장비정비,항법
778,2018,탄도탄조기경보레이더 부품구매,장비,레이더
657,2018,주파수교란장비(GRIFFIN) 부품정비,장비정비,통신
2663,2024,KC-330 공지통신무전기 성능개량사업,장비,통신


## 3. 배경 ⓪용 점검 — 연도별 예산·계약 확정분·상위 3건 (해석은 하지 않고 표만 남긴다)

`예산금액`은 집행 예정액(계획)이며, 2019·2025년 급증이 소수 대형 건 때문인지 확인하기 위한 표다. 판단번호 단위 값이라 원본 3,029행 합과 5행만큼 다르다.


In [6]:
yr = p.groupby("plan_year").agg(plan_count=("decision_no", "size"), budget_100m=("budget_krw", lambda s: round(s.sum()/1e8)),
                                contracted=("is_contracted", "sum"),
                                contracted_budget_100m=("budget_krw", lambda s: round(s[p.loc[s.index, "is_contracted"]==1].sum()/1e8)),
                                elec_cand=("is_electronics_candidate", "sum"),
                                elec_cand_budget_100m=("budget_krw", lambda s: round(s[p.loc[s.index, "is_electronics_candidate"]==1].sum()/1e8)))
yr["elec_cand_budget_pct(잠정)"] = (yr["elec_cand_budget_100m"] / yr["budget_100m"] * 100).round(1)
display(yr)
top = (p.sort_values("budget_krw", ascending=False).groupby("plan_year").head(3)
         [["plan_year", "decision_no", "rep_item_name", "exec_type", "progress_status", "budget_krw", "system_family_hint"]].copy())
top["budget_100m"] = (top["budget_krw"] / 1e8).round(1)
top["share_of_year_pct"] = top.apply(lambda r: round(r.budget_krw / p.loc[p.plan_year == r.plan_year, "budget_krw"].sum() * 100, 1), axis=1)
display(top.drop(columns="budget_krw").sort_values(["plan_year", "budget_100m"], ascending=[True, False]).reset_index(drop=True))

,plan_count,budget_100m,contracted,contracted_budget_100m,elec_cand,elec_cand_budget_100m,elec_cand_budget_pct(잠정)
plan_year,,,,,,,
2017,405,14278,0,0,55,4084,28.6
2018,469,10930,0,0,39,2602,23.8
2019,469,24808,338,23468,38,16109,64.9
2020,341,16663,200,14408,49,5631,33.8
2021,282,19127,199,17506,44,2738,14.3
2022,295,9015,201,7885,46,838,9.3
2023,244,21385,163,20177,45,1930,9.0
2024,232,16962,123,13647,34,1607,9.5
2025,286,49413,179,41680,42,3387,6.9


,plan_year,decision_no,rep_item_name,exec_type,progress_status,system_family_hint,budget_100m,share_of_year_pct
0,2017,DAH47039001,F-15K 항공기 PBL 2차사업,기타,2차공고의뢰중,NaN,4040.0,28.3
1,2017,DAE97268001,E-737 기술지원,기술용역,2차공고의뢰중,NaN,1170.0,8.2
2,2017,PDA97277002,가변심도음탐기,장비,2차공고의뢰중,음탐,885.0,6.2
3,2018,DFA98448001,탄도탄조기경보레이더 부품구매,장비,2차공고의뢰중,레이더,1570.0,14.4
4,2018,PCA98138001,소해함후속함 복합식소해장비,장비,2차공고의뢰중,NaN,922.3,8.4
5,2018,DAE98001001,E-737 기술지원,기술용역,2차공고의뢰중,NaN,836.6,7.7
6,2019,WSE99948001,대포병탐지레이더 기술용역,기술용역,계약,레이더,15000.0,60.5
7,2019,DDAR0001001,공중전투기동훈련용탑재장비,장비,계약,NaN,938.4,3.8
8,2019,DAMB9981001,CN-235 기술도서,한도액부품,계약,NaN,715.9,2.9
9,2020,DAAT0003002,E-737 IFF 및 LINK-16 성능개량사업,장비,계약,통신,4690.0,28.1


## 4. 팀 서버 적재 (대상 테이블이 비어 있을 때만) + `meta_load_log`


In [7]:
def count(t):
    cur.execute(f"SELECT COUNT(*) FROM {t}"); return cur.fetchone()[0]
def py(v):
    if v is None: return None
    if isinstance(v, float) and pd.isna(v): return None
    if hasattr(v, "item") and not isinstance(v, str): return v.item()
    return v
NOW = datetime.now().replace(microsecond=0)

if count("clean_dapa_localized_item") > 0:
    print("clean_dapa_localized_item 비어 있지 않음 → 건너뜀 (재적재는 db/reset_data.sql 또는 TRUNCATE 후)")
else:
    log_raw_stage(cur, "raw_dapa_localized_item", raw("raw_dapa_localized_item"), USER)   # 원본 전체 단계(파일 파서 행 수)
    cols = ["project_name", "part_mgmt_no", "nsn", "fsc4", "fsc2", "item_name", "contractor_name", "contractor_name_norm",
            "last_modified_date", "dup_count", "is_electronic_group", "first_raw_row_id", "cleaned_at"]
    b2["cleaned_at"] = NOW
    rows = [tuple(py(v) for v in r) for r in b2[cols].itertuples(index=False, name=None)]
    sql = f"INSERT INTO clean_dapa_localized_item ({','.join(cols)}) VALUES ({','.join(['%s']*len(cols))})"
    for i in range(0, len(rows), 2000):
        cur.executemany(sql, rows[i:i+2000])
    log_stage(cur, "dapa_localized_item", "clean_dapa_localized_item", "중복 처리 후", "사업명×부품관리번호 고유", len(b2),
              f"완전 중복 {len(raw_b2)-len(b2)}행을 dup_count로 축약(원본 보존). 같은 키 내용 충돌 {int(b2['has_conflict'].sum())}건은 대표 행(row_id 최소) 사용",
              f"{NB_TAG} §1 (pandas groupby first)", USER)
    conn.commit()
    print("clean_dapa_localized_item 적재:", count("clean_dapa_localized_item"))

if count("clean_dapa_overseas_plan") > 0:
    print("clean_dapa_overseas_plan 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_dapa_overseas_plan", raw("raw_dapa_overseas_plan"), USER)   # 원본 전체 단계(파일 파서 행 수)
    cols = ["decision_no", "plan_year", "plan_month", "rep_item_name", "exec_type", "contract_method", "exec_agency", "budget_krw",
            "progress_status", "is_contracted", "is_electronics_candidate", "electronics_review_status", "system_family_hint",
            "dup_count", "has_conflict", "first_raw_row_id", "cleaned_at"]
    p["cleaned_at"] = NOW
    rows = [tuple(py(v) for v in r) for r in p[cols].itertuples(index=False, name=None)]
    sql = f"INSERT INTO clean_dapa_overseas_plan ({','.join(cols)}) VALUES ({','.join(['%s']*len(cols))})"
    cur.executemany(sql, rows)
    log_stage(cur, "dapa_overseas_plan", "clean_dapa_overseas_plan", "중복 처리 후", "판단번호 단위", len(p),
              f"필수값 결측 {len(bad)}행 제외(row_id {bad['row_id'].tolist()}), 같은 판단번호 {int((p['dup_count']>1).sum())}쌍은 PK 축약(dup_count, 내용 충돌 {int(p['has_conflict'].sum())}건, 빠지는 예산 {round(dropped_budget/1e8,1)}억원)",
              f"{NB_TAG} §2", USER)
    log_stage(cur, "dapa_overseas_plan", "clean_dapa_overseas_plan", "관련 후보", "대표품명 키워드 전자 관련 후보(잠정, 미검수)", n_cand,
              "키워드 미일치 제외. 검수 전이라 확정 아님(electronics_review_status=미검수)", f"{NB_TAG} §2", USER)
    conn.commit()
    print("clean_dapa_overseas_plan 적재:", count("clean_dapa_overseas_plan"))

clean_dapa_localized_item 비어 있지 않음 → 건너뜀 (재적재는 db/reset_data.sql 또는 TRUNCATE 후)
clean_dapa_overseas_plan 비어 있지 않음 → 건너뜀


## 5. 적재 후 검증 (뷰가 실제로 값을 내는지)


In [8]:
display(q("SELECT 'clean_dapa_localized_item' t, COUNT(*) n, COUNT(DISTINCT part_mgmt_no) parts, SUM(dup_count) raw_rows FROM clean_dapa_localized_item "
          "UNION ALL SELECT 'clean_dapa_overseas_plan', COUNT(*), COUNT(DISTINCT decision_no), SUM(dup_count) FROM clean_dapa_overseas_plan"))
display(q("SELECT plan_year, SUM(plan_count) plan_count, ROUND(SUM(budget_krw)/1e8) budget_100m, SUM(contracted_count) contracted, "
          "SUM(elec_candidate_count) elec_cand, ROUND(SUM(elec_candidate_budget_krw)/1e8) elec_cand_budget_100m, ROUND(SUM(elec_confirmed_budget_krw)/1e8) elec_confirmed "
          "FROM v_overseas_plan_yearly GROUP BY plan_year ORDER BY plan_year"))
display(q("SELECT dataset_key, table_name, stage, stage_detail, row_count, exclusion_reason FROM meta_load_log WHERE table_name LIKE 'clean_%' ORDER BY log_id"))
conn.close()

,t,n,parts,raw_rows
0,clean_dapa_localized_item,25025,12788,33965
1,clean_dapa_overseas_plan,3023,3023,3028


,plan_year,plan_count,budget_100m,contracted,elec_cand,elec_cand_budget_100m,elec_confirmed
0,2017,405,14278.0,0,55,4084.0,0.0
1,2018,469,10930.0,0,39,2602.0,0.0
2,2019,469,24808.0,338,38,16109.0,0.0
3,2020,341,16663.0,200,49,5631.0,0.0
4,2021,282,19127.0,199,44,2738.0,0.0
5,2022,295,9015.0,201,46,838.0,0.0
6,2023,244,21385.0,163,45,1930.0,0.0
7,2024,232,16962.0,123,34,1607.0,0.0
8,2025,286,49413.0,179,42,3387.0,0.0


,dataset_key,table_name,stage,stage_detail,row_count,exclusion_reason
0,dapa_localized_item,clean_dapa_localized_item,중복 처리 후,사업명×부품관리번호 고유,25025,완전 중복 8940행을 dup_count로 축약(원본 보존). 같은 키 내용 충돌 ...
1,dapa_localized_item,clean_dapa_localized_item,관련 후보,ref_category_map 후보 FSC 대응(사업×부품),400,화이트리스트 HS 대응 후보 FSC(5820·5825·5840·5855·5860·5...
2,dapa_overseas_plan,clean_dapa_overseas_plan,중복 처리 후,판단번호 단위,3023,"필수값 결측 1행 제외(row_id [768]), 같은 판단번호 5쌍은 PK 축약(..."
3,dapa_overseas_plan,clean_dapa_overseas_plan,관련 후보,"대표품명 키워드 전자 관련 후보(잠정, 미검수)",392,키워드 미일치 제외. 검수 전이라 확정 아님(electronics_review_st...
4,kdsis_nsn,clean_kdsis_nsn_ref,중복 처리 후,NSN×CAGE×참조번호 고유,225635,원본 완전 중복(source 열 제외) 제거
5,kdsis_nsn,clean_kdsis_nsn,중복 처리 후,NSN 고유(숫자13 + 검토),135864,NSN별 대표 행 1개
6,dapa_contract,clean_dapa_contract,중복 처리 후,계약번호×차수(0→00) 고유,43111,"키 충돌 1건(2024UMM1504-01, 원본 완전 중복 2행)은 대표 행 1개만..."
7,dapa_contract,clean_dapa_contract,중복 처리 후,계약 단위(is_latest_seq=1),37608,변경계약 차수를 계약번호당 최종 차수 1행으로 축약(이력은 clean에 보존)
8,dapa_bid_notice,clean_dapa_bid_notice,중복 처리 후,참조공고번호+차수 고유(열 밀림 2행 제외),10840,열 밀림 원본 2행(LCF0223 1·2차)은 clean_excluded_row C...
9,dapa_bid_result,clean_dapa_bid_result,중복 처리 후,"공고번호+차수+result_seq(열 밀림 2행 제외, 중복 키 행 보존)",7403,열 밀림 2행 COL_SHIFT. 중복 199키는 삭제하지 않고 dup_kind로 ...


## 남은 일 (이 노트북이 하지 않은 것)

- 전자 관련 후보 표본 검수 → `electronics_review_status`를 `확정`/`오탐`으로 UPDATE(사용자). 검수 전 비중은 "잠정".
- 같은 키 내용 충돌 건의 대표 행 선택 근거 확인(출력값 참고).
- `clean_dapa_contract`·`clean_krit_task`·`clean_company*`는 별도 노트북.
